# Bab 2 — Research Problem Formulation

Notebook pendamping Bab 2 Level 7. Jalankan sel dari atas ke bawah. Bagian A sampai G mengikuti halaman bab. Bagian **Praktik** di bawahnya berisi kegiatan yang tidak ada di halaman bab: kamu menulis pertanyaan risetmu sendiri, memperluas pemeriksa, dan mengisi lembar rumusan masalah.

Dataset: Breast Cancer Wisconsin (Diagnostic), UCI, CC BY 4.0. Bukan alat diagnosis.

## Learning Objectives

Setelah mengerjakan notebook ini, kamu mampu:

- Memeriksa kelayakan data sebelum merumuskan pertanyaan.
- Menulis pertanyaan riset sendiri dan memeriksanya dengan daftar periksa.
- Memperluas daftar periksa dan melihat kapan aturan baru menolong atau menyesatkan.
- Mengisi lembar rumusan masalah dan menemukan bagian yang masih kosong.

## Langkah 0: Pindah ke folder utama proyek

In [1]:
import os
import sys
from pathlib import Path

if not Path("data").exists() and not Path("examples").exists():
    os.chdir("../..")

ROOT = Path.cwd()
# alat bantu bersama Level 7 (examples/level-7/_research.py)
sys.path.insert(0, str(Path("examples/level-7").resolve()))

## Bagian A: Siapkan alat

In [2]:
import json
import re

import pandas as pd

import _research as rs

X_all, diagnosis = rs.load_wdbc()

## Bagian B: Apakah data bisa menjawab?

In [3]:
# Sebelum merumuskan pertanyaan, periksa apa yang sebenarnya dimiliki data.
facts = {
    "jumlah_sampel": X_all.shape[0],
    "jumlah_fitur": X_all.shape[1],
    "nilai_hilang": int(X_all.isna().sum().sum()),
    "ada_label_eksternal": diagnosis is not None,
    "kelas_label": diagnosis.value_counts().to_dict(),
    "ada_kolom_waktu": any("time" in c or "date" in c for c in X_all.columns),
    "ada_kolom_pasien": any("id" == c.lower() or "patient" in c.lower() for c in X_all.columns),
}
for key, value in facts.items():
    print(f"{key:22s}: {value}")

jumlah_sampel         : 569
jumlah_fitur          : 30
nilai_hilang          : 0
ada_label_eksternal   : True
kelas_label           : {'benign': 357, 'malignant': 212}
ada_kolom_waktu       : False
ada_kolom_pasien      : False


## Bagian C: Pemeriksa pertanyaan riset

In [4]:
def check_question(text):
    """Daftar periksa sederhana. True berarti ada MASALAH pada aspek tersebut.

    Ini alat bantu berpikir berbasis kata kunci, bukan penilai. Ia bisa salah (lihat contoh)."""
    low = text.lower()
    words = set(re.findall(r"[a-z]+", low))
    return {
        "bukan_pertanyaan": not text.strip().endswith("?"),
        "mulai_dari_algoritma": low.startswith(("menggunakan", "menerapkan", "memakai")),
        "klaim_sebab_akibat": any(w in low for w in ["menyebabkan", "penyebab", "akibat", "mengapa"]),
        "klaim_klinis": any(w in low for w in ["mendiagnosis", "mendeteksi", "memprediksi", "mengobati"]),
        "klaim_universal": any(w in low for w in ["selalu", "semua data", "pasti", "tidak pernah"]),
        "tidak_terukur": not (words & {"stabil", "seberapa", "ari", "nmi", "silhouette", "berubah", "selaras", "berbeda"}),
    }


def verdict(flags):
    problems = [name for name, is_problem in flags.items() if is_problem]
    return "lolos" if not problems else "perlu diperbaiki: " + ", ".join(problems)

## Bagian D: Menguji pertanyaan baik dan buruk

In [5]:
examples = {
    "B1": "Menggunakan K-Means untuk mengelompokkan data kanker.",
    "B2": "Apakah clustering bisa mendiagnosis kanker payudara?",
    "B3": "Mengapa ukuran inti sel menyebabkan kelompok ganas?",
    "B4": "Algoritma clustering mana yang selalu terbaik untuk semua data?",
    "B5": "Bagaimana data ini?",
    "G1": "Seberapa stabil pengelompokan K-Means pada 30 fitur morfologi bila seed dan sampel data diubah?",
    "G2": "Apakah pengelompokan pada 30 fitur morfologi berubah bila fitur tidak distandarkan, dan seberapa besar perubahannya (ARI)?",
    "G3": "Seberapa selaras pengelompokan tak terawasi pada profil morfologi dengan diagnosis yang tercatat (ARI dan NMI)?",
    "T1": "Apakah hasil clustering stabil dan selaras dengan diagnosis serta berbeda antar algoritma pada semua aspek?",
}
for code, text in examples.items():
    print(f"{code}: {text}")
    print(f"    -> {verdict(check_question(text))}")

B1: Menggunakan K-Means untuk mengelompokkan data kanker.
    -> perlu diperbaiki: bukan_pertanyaan, mulai_dari_algoritma, tidak_terukur
B2: Apakah clustering bisa mendiagnosis kanker payudara?
    -> perlu diperbaiki: klaim_klinis, tidak_terukur
B3: Mengapa ukuran inti sel menyebabkan kelompok ganas?
    -> perlu diperbaiki: klaim_sebab_akibat, tidak_terukur
B4: Algoritma clustering mana yang selalu terbaik untuk semua data?
    -> perlu diperbaiki: klaim_universal, tidak_terukur
B5: Bagaimana data ini?
    -> perlu diperbaiki: tidak_terukur
G1: Seberapa stabil pengelompokan K-Means pada 30 fitur morfologi bila seed dan sampel data diubah?
    -> lolos
G2: Apakah pengelompokan pada 30 fitur morfologi berubah bila fitur tidak distandarkan, dan seberapa besar perubahannya (ARI)?
    -> lolos
G3: Seberapa selaras pengelompokan tak terawasi pada profil morfologi dengan diagnosis yang tercatat (ARI dan NMI)?
    -> lolos
T1: Apakah hasil clustering stabil dan selaras dengan diagnosis serta

## Bagian E: Mencocokkan pertanyaan dengan pendekatan

In [6]:
matching = pd.DataFrame(
    [
        ["Eksploratif", "Adakah struktur kelompok?", "K-Means, Hierarchical", "Silhouette, visualisasi", "Level 4 B2-B3, Level 6 B8"],
        ["Komparatif", "Algoritma mana yang cocok untuk data ini?", "K-Means, GMM, DBSCAN, Hierarchical", "Metrik di ruang yang sama", "Level 4 B7, Level 6 B4"],
        ["Sensitivitas", "Seberapa peka hasil terhadap praproses atau parameter?", "Satu algoritma, satu faktor diubah", "ARI terhadap baseline", "Level 6 B2, B3, B5"],
        ["Kestabilan", "Apakah hasil tahan terhadap seed dan sampel?", "Satu algoritma, ulang berkali-kali", "ARI antar-ulangan, Jaccard", "Level 6 B9"],
        ["Validasi eksternal", "Seberapa selaras dengan pengetahuan di luar data?", "Clustering dulu, label sesudahnya", "ARI, NMI terhadap label", "Level 5 B8"],
    ],
    columns=["Tipe pertanyaan", "Contoh", "Pendekatan", "Evaluasi", "Rujukan"],
)
print(matching.to_string(index=False))

   Tipe pertanyaan                                                 Contoh                         Pendekatan                   Evaluasi                   Rujukan
       Eksploratif                              Adakah struktur kelompok?              K-Means, Hierarchical    Silhouette, visualisasi Level 4 B2-B3, Level 6 B8
        Komparatif              Algoritma mana yang cocok untuk data ini? K-Means, GMM, DBSCAN, Hierarchical  Metrik di ruang yang sama    Level 4 B7, Level 6 B4
      Sensitivitas Seberapa peka hasil terhadap praproses atau parameter? Satu algoritma, satu faktor diubah      ARI terhadap baseline        Level 6 B2, B3, B5
        Kestabilan           Apakah hasil tahan terhadap seed dan sampel? Satu algoritma, ulang berkali-kali ARI antar-ulangan, Jaccard                Level 6 B9
Validasi eksternal      Seberapa selaras dengan pengetahuan di luar data?  Clustering dulu, label sesudahnya    ARI, NMI terhadap label                Level 5 B8


## Bagian F: Rumusan masalah proyek

In [7]:
# Rumusan masalah untuk proyek akhir (Bab 10). Ini DRAF: dirumuskan ulang di Bab 3 setelah tinjauan pustaka.
project = {
    "judul_kerja": "Stabilitas dan keselarasan struktur kelompok pada profil morfologi inti sel",
    "masalah": "Algoritma clustering berbasis partisi menghasilkan kelompok pada data apa pun, tetapi belum tentu kelompok itu stabil atau bermakna.",
    "tujuan": [
        "Mengukur kestabilan pengelompokan terhadap seed dan sampel data.",
        "Mengukur kepekaan pengelompokan terhadap penyamaan skala dan pilihan algoritma.",
        "Mengukur keselarasan kelompok dengan diagnosis yang tercatat sebagai validasi eksternal.",
    ],
    "pertanyaan": {
        "RQ1": "Seberapa stabil pengelompokan K-Means pada 30 fitur morfologi bila seed dan sampel data diubah?",
        "RQ2": "Apakah pengelompokan pada 30 fitur morfologi berubah bila fitur tidak distandarkan, dan seberapa besar perubahannya (ARI)?",
        "RQ3": "Seberapa selaras pengelompokan tak terawasi pada profil morfologi dengan diagnosis yang tercatat (ARI dan NMI)?",
    },
    "dalam_lingkup": ["satu dataset publik", "empat algoritma dari Level 4", "metrik dari Level 5", "30 fitur numerik"],
    "di_luar_lingkup": ["klaim klinis atau diagnosis", "klaim sebab-akibat", "data pasien baru", "generalisasi ke dataset lain"],
    "keterbatasan": [
        "satu dataset dengan 569 sampel",
        "label diagnosis hanya dua kelas dan bukan satu-satunya struktur yang mungkin ada",
        "hasil berlaku untuk pengaturan yang diuji",
    ],
    "kontribusi": "didaktik dan metodologis: mendemonstrasikan protokol penelitian clustering yang bisa diulang. Tidak mengklaim kebaruan ilmiah.",
}
for rq_id, text in project["pertanyaan"].items():
    print(f"{rq_id}: {verdict(check_question(text))}")

RQ1: lolos
RQ2: lolos
RQ3: lolos


## Bagian G: Menyimpan rumusan masalah

In [8]:
project["data"] = {"sumber": "UCI, Breast Cancer Wisconsin (Diagnostic), CC BY 4.0", "sidik_jari": rs.dataset_fingerprint(X_all)}
project["status"] = "draf Bab 2, dirumuskan ulang di Bab 3"
rs.RESULTS_DIR.mkdir(exist_ok=True)
out = rs.RESULTS_DIR / "02_research_problem.json"
out.write_text(json.dumps(project, indent=2, ensure_ascii=False), encoding="utf-8")
print("Disimpan:", out.relative_to(ROOT).as_posix())
print("Jumlah pertanyaan:", len(project["pertanyaan"]), "| jumlah tujuan:", len(project["tujuan"]))

Disimpan: examples/level-7/results/02_research_problem.json
Jumlah pertanyaan: 3 | jumlah tujuan: 3


## Praktik 1: Tulis pertanyaan risetmu

Ganti dua kalimat di bawah dengan pertanyaan milikmu sendiri (untuk dataset ini), lalu jalankan sel. Contoh bawaan sengaja belum sempurna. Coba perbaiki sampai lolos, lalu tanyakan pada dirimu: apakah lolos berarti sudah matang?

In [9]:
my_questions = {
    "Saya-1": "Apakah clustering bagus untuk data ini?",
    "Saya-2": "Seberapa berubah pengelompokan K-Means (ARI) bila K diganti dari 2 menjadi 3?",
}
for code_, text in my_questions.items():
    print(f"{code_}: {text}")
    print(f"    -> {verdict(check_question(text))}")

Saya-1: Apakah clustering bagus untuk data ini?
    -> perlu diperbaiki: tidak_terukur
Saya-2: Seberapa berubah pengelompokan K-Means (ARI) bila K diganti dari 2 menjadi 3?
    -> lolos


## Praktik 2: Perluas pemeriksa

Kita tambahkan satu aturan: pertanyaan yang memuat kata sambung "dan" atau "serta" dua kali atau lebih dianggap memuat banyak aspek. Lalu bandingkan hasil versi lama dan baru pada semua contoh bab. Perhatikan contoh mana yang berubah.

In [10]:
import re

def check_question_v2(text):
    flags = check_question(text)
    tokens = re.findall(r"[a-z]+", text.lower())
    flags["banyak_aspek"] = (tokens.count("dan") + tokens.count("serta")) >= 2
    return flags

rows = []
for code_, text in {**examples, **my_questions}.items():
    rows.append({"kode": code_, "v1": verdict(check_question(text)).split(":")[0], "v2": verdict(check_question_v2(text)).split(":")[0]})
comparison = pd.DataFrame(rows).set_index("kode")
print(comparison.to_string())
print("Berubah antara v1 dan v2:", comparison.index[comparison["v1"] != comparison["v2"]].tolist())

                      v1                v2
kode                                      
B1      perlu diperbaiki  perlu diperbaiki
B2      perlu diperbaiki  perlu diperbaiki
B3      perlu diperbaiki  perlu diperbaiki
B4      perlu diperbaiki  perlu diperbaiki
B5      perlu diperbaiki  perlu diperbaiki
G1                 lolos             lolos
G2                 lolos             lolos
G3                 lolos             lolos
T1                 lolos  perlu diperbaiki
Saya-1  perlu diperbaiki  perlu diperbaiki
Saya-2             lolos             lolos
Berubah antara v1 dan v2: ['T1']


## Praktik 3: Lembar rumusan masalah

Isi nilai `"..."` di bawah dengan rumusan masalahmu sendiri, lalu jalankan sel. Fungsi `completeness` melaporkan bagian yang masih kosong. Tujuannya melatih kebiasaan memeriksa kelengkapan sebelum lanjut ke bab berikutnya.

In [11]:
sheet = {
    "masalah": "...",
    "tujuan": ["..."],
    "pertanyaan": {"RQ1": "..."},
    "di_luar_lingkup": ["klaim klinis"],
    "keterbatasan": ["..."],
    "kontribusi": "...",
}

def completeness(record):
    empty = []
    for key, value in record.items():
        items = list(value.values()) if isinstance(value, dict) else (value if isinstance(value, list) else [value])
        if any(str(item).strip() in ("", "...") for item in items):
            empty.append(key)
    return empty

print("Bagian yang belum diisi:", completeness(sheet))

Bagian yang belum diisi: ['masalah', 'tujuan', 'pertanyaan', 'keterbatasan', 'kontribusi']


## Latihan

1. Perbaiki `Saya-1` sampai lolos pemeriksa, lalu tulis satu kalimat tentang apa yang masih kurang meski sudah lolos.
2. Tambahkan aturan yang mendeteksi kata "semua" sebagai klaim universal. Apakah ada pertanyaan baik yang ikut tertandai?
3. Isi `sheet` sampai `completeness(sheet)` mengembalikan daftar kosong. Apakah pertanyaanmu sudah lolos `check_question_v2`?
4. Dengan dataset ini, apa satu pertanyaan penting yang **tidak bisa** dijawab? Jelaskan dari hasil pemeriksaan kelayakan di Bagian B.

## Ringkasan

- Pertanyaan riset menentukan data, fitur, algoritma, dan evaluasi.
- Periksa kelayakan data **sebelum** merumuskan pertanyaan.
- Daftar periksa otomatis hanyalah alat bantu: aturan baru bisa menangkap satu kesalahan dan melewatkan atau salah menandai yang lain.
- Lembar rumusan masalah yang lengkap (masalah, tujuan, pertanyaan, lingkup, keterbatasan, kontribusi) adalah syarat sebelum tinjauan pustaka di Bab 3.